# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soumyajeetrc/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Feature Vector Setup:
We are pulling a sample of data to engineer five core features that represent a page's visibility and engagement. We define our target label TARGET_bad_page as any page dropping below 10 daily impressions.

In [3]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

print("--- 1. BUILDING THE FEATURE VECTOR ---")
my_token = userdata.get('HF_TOKEN')
stream_data = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=my_token,
    streaming=True
)

# Pull a working sample
df = pd.DataFrame(list(stream_data.take(5000)))

# Build 5 Features
df['feature_position'] = df['gsc_avg_position']
df['feature_ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, 1)
df['feature_clicks'] = df['gsc_clicks']
df['feature_has_clicks'] = (df['gsc_clicks'] > 0).astype(int)
df['feature_impressions'] = df['gsc_impressions'] # We will use this to spring the trap!

# Define the Target
df['TARGET_bad_page'] = (df['gsc_impressions'] < 10).astype(int)

print(f"Engineered features for {len(df)} rows.")
display(df[['feature_position', 'feature_ctr', 'feature_clicks', 'feature_has_clicks', 'feature_impressions', 'TARGET_bad_page']].head())

--- 1. BUILDING THE FEATURE VECTOR ---


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Engineered features for 5000 rows.


,feature_position,feature_ctr,feature_clicks,feature_has_clicks,feature_impressions,TARGET_bad_page
0,3.833333,0.0,0,0,30,0
1,71.600000,0.0,0,0,5,1
2,34.000000,0.0,0,0,1,1
3,23.333333,0.0,0,0,6,1
4,17.800000,0.0,0,0,5,1


## 2. Feature notes (meaning, missing, categorical, available-when?)
Feature Dictionary & Availability:

feature_position: Google Search ranking. Missing values treated as 100 (unranked). Available at the decision moment.

feature_ctr: Clicks divided by impressions. Missing values (0 impressions) output as 0. Available at the decision moment.

feature_clicks: Total raw clicks. Available at the decision moment.

feature_has_clicks: Binary flag (1 if clicks > 0). Available at the decision moment.

feature_impressions: Raw impressions. Warning: Highly correlated with our target definition.

In [4]:
# Documentation completed above.


## 3. The leakage hunt
The Leakage Trap:
Our target is defined as impressions < 10. If we include feature_impressions as a predictive feature, the model isn't learning; it is just memorizing the answer key. We will train it with the leak, observe the falsely perfect score, and then document why this feature must be destroyed.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split

print("--- 3. THE LEAKAGE HUNT ---")
features_leaked = ['feature_position', 'feature_ctr', 'feature_clicks', 'feature_has_clicks', 'feature_impressions']

X_train, X_test, y_train, y_test = train_test_split(df[features_leaked], df['TARGET_bad_page'], test_size=0.2, random_state=42)

# Train the cheating model
trap_model = DecisionTreeClassifier(random_state=42)
trap_model.fit(X_train, y_train)
trap_predictions = trap_model.predict(X_test)
trap_precision = precision_score(y_test, trap_predictions, zero_division=0)

print(f"🚨 SCORE WITH LEAKED FEATURE: {trap_precision:.1%}")
print("Result: The score is impossibly high. The model cheated by using 'feature_impressions' to predict an impression-based target.")
print("Action: 'feature_impressions' must be dropped before production.")


--- 3. THE LEAKAGE HUNT ---
🚨 SCORE WITH LEAKED FEATURE: 100.0%
Result: The score is impossibly high. The model cheated by using 'feature_impressions' to predict an impression-based target.
Action: 'feature_impressions' must be dropped before production.


## 4. What I excluded and why
Excluded Fields:

client_id and client_hash_id: Excluded to prevent the model from learning client-specific biases and to maintain strict data privacy boundaries.

url / page_path: Excluded because raw text strings cannot be processed by a standard decision tree without NLP vectorization, and they pose a PII risk.

In [6]:
# Exclusions documented above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.